<a href="https://colab.research.google.com/github/intisariapps-com/intiVoice_Studio/blob/main/intiVoice_Studio_WebUI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 🎙️ intiVoice AI — Web Studio Visual Mandiri (Gradio Edition V1.6.3)

> ⏰ **Terakhir Diperbarui:** 21 September 2026 | **Rilis:** V1.6.3 (Native In-Memory Production Engine)
> 🖼️ **Interactive Inline UI:** Antarmuka visual interaktif langsung tampil di bawah sel Colab (*embedded iframe*).
> 🌐 **Public URL Tunnel Otomatis:** Menghasilkan link publik acak gratis (`https://xxxx.gradio.live`) tanpa perlu akun Cloudflare atau token.
> ⚡ **Direct GPU In-Memory Execution:** Bebas dari koneksi port 8000 / HTTP socket. 100% bebas error *Connection Refused*.
> 🔐 **Fitur Lengkap Produksi:** Didukung Smart Chunking, Voice Anchor DNA, 13 Emosi Vokal, Dual Subtitle SRT/ASS, dan Dual-Speaker Dialogue.

Notebook ini adalah **Antarmuka Mandiri (All-in-One)** untuk Text-to-Speech (TTS), Voice Design, Ultimate Voice Cloning 1:1, dan Percakapan Dua Pembicara berkualitas **48kHz Studio Audio**.

### ✨ Fitur Utama V1.6.3:
1. 🗣️ **Human-Cadence Smart Chunking**: Naskah panjang dipotong rapi per kalimat (180–240 karakter) dengan jeda nafas alami, bebas risiko terpotong.
2. 🔒 **Voice Anchor DNA System**: Mengunci konsistensi warna vokal sepanjang naskah panjang tanpa pergeseran nada.
3. 🎭 **Selector 13 Emosi Vokal**: Emosi alami (Berbisik, Tertawa, Sedih, Marah, Khawatir, Bahagia, dsb) langsung aktif pada vokal target.
4. 👥 **Dual-Speaker Dialogue Studio**: Render percakapan dua tokoh berbeda suara sekaligus subtitle dialog tersinkronisasi.
5. 🎬 **Ekspor 3-Berkas Sekaligus**: Unduh audio **WAV 48kHz**, subtitle **CapCut (.SRT)**, dan subtitle **Karaoke 9:16 (.ASS)** dengan 1 klik.

---
### 🚀 Cara Menjalankan:
1. Pastikan runtime GPU aktif (**Runtime** → **Change runtime type** → **T4 GPU**).
2. Klik tombol **Play (▶)** pada sel kode di bawah ini.
3. Tunggu ~45 detik hingga model siap. Antarmuka Web Studio interaktif akan otomatis muncul di bawah sel, dan link publik `https://xxxx.gradio.live` akan tercetak di layar.


In [ ]:
"""
🎙️ INTIVOICE AI — STANDALONE GRADIO STUDIO (V1.6.3 NATIVE IN-MEMORY)
Hak Cipta (C) 2026 IntisariApps.com. Seluruh hak cipta dilindungi.
"""

# @title 🚀 LUNCURKAN WEB STUDIO VISUAL (1-KLIK)
# @markdown Jalankan sel ini untuk memulai Web Studio interaktif di layar Colab dan membuat link publik gratis.

import os
import sys
import time
import subprocess
import re
import io
import random
import tempfile
from typing import Optional, List, Dict, Any, Tuple

print("=" * 80)
print("🎙️ MEMULAI INTIVOICE AI — STANDALONE GRADIO STUDIO V1.6.3")
print("=" * 80)

# 1. Verifikasi Akselerator GPU
gpu_check = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu_check.returncode != 0:
    print("⚠️ PERINGATAN: Runtime GPU tidak terdeteksi!")
    print("👉 Buka menu 'Runtime' -> 'Change runtime type' -> Pilih 'T4 GPU' lalu jalankan ulang.")
else:
    print("✅ Akselerator GPU Aktif & Siap Digunakan.")

# 2. Instalasi Dependensi Cepat via uv (~3 detik)
print("\n[1/3] ⚡ Memeriksa & memasang dependensi (Ultra-Fast via uv)...")
try:
    import voxcpm
    import soundfile as sf
    import gradio as gr
    import torch
    import torchaudio
    import numpy as np
    print("✅ Seluruh pustaka audio & Gradio sudah aktif di memori!")
except ImportError:
    print("⏳ Memasang voxcpm, soundfile, gradio, torchaudio via uv...")
    try:
        subprocess.check_call(["uv", "pip", "install", "--system", "voxcpm", "soundfile", "gradio", "ipython", "torchaudio", "librosa"])
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "voxcpm", "soundfile", "gradio", "ipython", "torchaudio", "librosa"])
    
    import voxcpm
    import soundfile as sf
    import gradio as gr
    import torch
    import torchaudio
    import numpy as np
    print("✅ Dependensi berhasil dipasang sempurna!")

from voxcpm import VoxCPM

# 3. Muat Model VoxCPM2 ke GPU
print("\n[2/3] 🧠 Memuat Model VoxCPM2 ke GPU...")
start_load = time.time()
model = VoxCPM.from_pretrained("openbmb/VoxCPM2", load_denoiser=False)
sample_rate = getattr(model.tts_model, "sample_rate", 48000)
print(f"✅ Model VoxCPM2 siap di GPU dalam {time.time() - start_load:.2f} detik! (Sample Rate: {sample_rate}Hz)")

# ------------------------------------------------------------------------------
# 4. LOGIKA ENGINE PRODUKSI NATIVE IN-MEMORY (FITUR PENUH ZIP)
# ------------------------------------------------------------------------------
EMOTION_THESAURUS = {
    "sedih": "sad tone",
    "menangis": "crying, emotional voice",
    "marah": "angry, firm projection",
    "senang": "happy, cheerful tone",
    "gembira": "happy, cheerful tone",
    "bahagia": "warm, joyful tone",
    "ceria": "cheerful, bright tone",
    "tertawa": "cheerful laughing tone",
    "antusias": "enthusiastic, energetic tone",
    "semangat": "enthusiastic, energetic tone",
    "bisik": "whispering softly, intimate voice",
    "berbisik": "whispering softly, intimate voice",
    "takut": "fearful, nervous tone",
    "panik": "panicked, hurried tone",
    "kaget": "surprised, shocked tone",
    "terkejut": "surprised, shocked tone",
    "tegas": "confident, firm authoritative tone",
    "wibawa": "confident, firm authoritative tone",
    "lelah": "tired, quiet soft voice",
    "romantis": "warm, soft romantic tone",
    "tenang": "calm tone, natural pacing",
}

def format_timestamp_srt(seconds: float) -> str:
    hrs = int(seconds // 3600)
    mins = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    millis = int(round((seconds - int(seconds)) * 1000))
    if millis >= 1000:
        millis = 999
    return f"{hrs:02d}:{mins:02d}:{secs:02d},{millis:03d}"

def format_timestamp_ass(seconds: float) -> str:
    hrs = int(seconds // 3600)
    mins = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    centis = int(round((seconds - int(seconds)) * 100))
    if centis >= 100:
        centis = 99
    return f"{hrs:01d}:{mins:02d}:{secs:02d}.{centis:02d}"

def get_adaptive_pause_sec(text: str) -> float:
    t = text.strip()
    if not t:
        return 0.35
    last_char = t[-1]
    if last_char in ['.', '!', '?', '。', '！', '？']:
        return 0.50
    elif last_char in [',', ';', ':', '-', '—', '，', '、']:
        return 0.25
    elif '\n' in text:
        return 0.70
    return 0.40

def split_text_into_smart_chunks(text: str, max_chars_per_chunk: int = 240) -> List[str]:
    clean_text = text.replace('\r\n', '\n').replace('\r', '\n').strip()
    if len(clean_text) <= max_chars_per_chunk:
        return [clean_text] if clean_text else []
    sentence_pattern = r'(?<=[.!?。！？\n])\s+'
    raw_sentences = [s.strip() for s in re.split(sentence_pattern, clean_text) if s.strip()]
    chunks, current_chunk = [], ""
    for s in raw_sentences:
        if len(current_chunk) + len(s) + 1 <= max_chars_per_chunk:
            current_chunk = f"{current_chunk} {s}".strip()
        else:
            if current_chunk:
                chunks.append(current_chunk)
            if len(s) > max_chars_per_chunk:
                clause_pattern = r'(?<=[,;:\-—，、])\s+'
                clauses = [c.strip() for c in re.split(clause_pattern, s) if c.strip()]
                sub_chunk = ""
                for c in clauses:
                    if len(sub_chunk) + len(c) + 1 <= max_chars_per_chunk:
                        sub_chunk = f"{sub_chunk} {c}".strip()
                    else:
                        if sub_chunk:
                            chunks.append(sub_chunk)
                        sub_chunk = c
                if sub_chunk:
                    chunks.append(sub_chunk)
                current_chunk = ""
            else:
                current_chunk = s
    if current_chunk:
        chunks.append(current_chunk)
    return chunks

def apply_fade(audio: np.ndarray, fade_samples: int = 480) -> np.ndarray:
    result = audio.copy()
    fs = min(fade_samples, len(result) // 4)
    if fs > 0:
        fade = np.linspace(0.0, 1.0, fs, dtype=np.float32)
        result[:fs] *= fade
        result[-fs:] *= fade[::-1]
    return result

def speed_to_control_hint(speed: float) -> str:
    if speed < 0.70:
        return "very slowly and clearly"
    elif speed < 0.88:
        return "slowly"
    elif speed > 1.15:
        return "fast"
    elif speed > 1.30:
        return "very fast"
    return ""

def parse_dialogue_chunk(text: str) -> Tuple[str, str, str]:
    m = re.match(r'^([^:\n]{1,25}):\s*(.*)$', text.strip(), flags=re.DOTALL)
    if not m:
        return "", "", text.strip()
    spk, content = m.group(1).strip(), m.group(2).strip()
    em_match = re.match(r'^\(([^)]+)\)\s*(.*)$', content, flags=re.DOTALL)
    if em_match:
        return spk, em_match.group(1).strip(), em_match.group(2).strip()
    return spk, "", content

# Presets Karakter Resmi & Kamus Emosi
PRESETS = {
    "🌸 Wanita Lembut & Ramah (Customer Service / Storytelling)": "A young Indonesian woman, gentle, sweet and very friendly voice, clear and soothing tone",
    "🎙️ Pria Narator Berwibawa (Dokumenter / E-Learning / Iklan)": "A mature Indonesian male narrator, deep warm authoritative voice, calm and professional pacing",
    "🎧 Host Podcast Santai & Energik (YouTube / TikTok / Sosmed)": "An energetic young Indonesian male podcast host, cheerful, expressive and casual tone",
    "📢 Pembaca Berita Resmi (News Anchor / Formal)": "A professional female news anchor, formal Indonesian intonation, clear articulation and confident pacing",
    "⚡ Kustom (Ketik Prompt Karakter Sendiri)": ""
}

EMOTIONS = [
    "(Netral / Standar)",
    "(berbisik)",
    "(tertawa)",
    "(senang)",
    "(sedih)",
    "(marah)",
    "(takut)",
    "(terkejut)",
    "(penasaran)",
    "(khawatir)",
    "(bangga)",
    "(lelah)",
    "(percaya diri)",
    "(tersenyum)"
]

OUTPUTS_DIR = "/tmp/intivoice_gradio_outputs"
os.makedirs(OUTPUTS_DIR, exist_ok=True)

# Helper Resample Audio Referensi ke 16kHz Mono
def prepare_reference_wav(audio_path: str) -> str:
    if not audio_path or not os.path.exists(audio_path):
        return ""
    try:
        waveform, sr = torchaudio.load(audio_path)
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
        if sr != 16000:
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
            waveform = resampler(waveform)
        tmp = tempfile.NamedTemporaryFile(suffix="_ref16k.wav", delete=False)
        torchaudio.save(tmp.name, waveform, 16000)
        return tmp.name
    except Exception as e:
        print(f"⚠️ Gagal resample audio referensi: {e}")
        return audio_path

# ==============================================================================
# CALLBACK TAB 1: STUDIO VOKAL TUNGGAL (NATIVE IN-MEMORY)
# ==============================================================================
def synthesize_single_gradio(text, preset_choice, custom_prompt, emotion_choice, ref_audio, ref_transcript, cfg_value, steps, seed, speed, progress=gr.Progress()):
    if not text or not text.strip():
        return None, None, None, "⚠️ Silakan masukkan naskah teks terlebih dahulu!"

    style = custom_prompt.strip() if preset_choice == "⚡ Kustom (Ketik Prompt Karakter Sendiri)" else PRESETS.get(preset_choice, "")
    clean_text = text.strip()
    if emotion_choice and emotion_choice != "(Netral / Standar)" and not clean_text.startswith("("):
        clean_text = f"{emotion_choice} {clean_text}"

    target_speed = float(speed or 1.0)
    speed_hint = speed_to_control_hint(target_speed)
    target_seed = int(seed) if seed is not None else 428190
    target_cfg = float(cfg_value or 2.0)
    inference_timesteps = int(steps or 10)

    # Siapkan Audio Referensi (Cloning)
    ref_16k_path = prepare_reference_wav(ref_audio) if ref_audio else None
    prompt_text_clean = ref_transcript.strip() if (ref_transcript and ref_transcript.strip()) else None
    is_ultimate_cloning = bool(ref_16k_path and prompt_text_clean)

    # Smart Sentence Chunking (Human-Cadence 240 chars)
    chunks = split_text_into_smart_chunks(clean_text, max_chars_per_chunk=240)
    num_chunks = len(chunks)
    if num_chunks == 0:
        return None, None, None, "⚠️ Naskah teks kosong setelah dibersihkan!"

    start_time = time.time()
    progress(0.05, desc=f"Menyiapkan {num_chunks} bagian naskah di GPU...")

    audio_segments = []
    chunk_durations = []
    voice_anchor_path = None
    use_voice_anchor = (ref_16k_path is None) and (num_chunks > 1)

    for i, chunk in enumerate(chunks):
        clean_c = re.sub(r'\s+', ' ', chunk.replace('\n', ' ')).strip()
        if not clean_c:
            continue

        _, em_tag, spoken = parse_dialogue_chunk(clean_c)
        if not spoken:
            spoken = clean_c

        # Emosi lokal & speed hint
        controls = []
        if em_tag and em_tag.lower() in EMOTION_THESAURUS:
            controls.append(EMOTION_THESAURUS[em_tag.lower()])
        elif style:
            controls.append(style)
        if speed_hint:
            controls.append(speed_hint)

        prefix = f"({', '.join(controls)})" if controls else ""
        prompt_chunk = f"{prefix}{spoken}"

        chunk_seed = target_seed + i
        torch.manual_seed(chunk_seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(chunk_seed)
        np.random.seed(chunk_seed)

        gen_kwargs = {
            "text": prompt_chunk,
            "cfg_value": target_cfg,
            "inference_timesteps": inference_timesteps,
            "retry_badcase": True,
        }

        if ref_16k_path:
            gen_kwargs["reference_wav_path"] = ref_16k_path
            if is_ultimate_cloning:
                gen_kwargs["prompt_wav_path"] = ref_16k_path
                gen_kwargs["prompt_text"] = prompt_text_clean
        elif voice_anchor_path:
            gen_kwargs["reference_wav_path"] = voice_anchor_path

        with torch.inference_mode():
            wav = model.generate(**gen_kwargs)

        # Voice Anchor DNA: kunci 3 detik pertama chunk 0
        if i == 0 and use_voice_anchor and voice_anchor_path is None:
            try:
                anchor_wav = wav[:int(sample_rate * 3)] if len(wav) > int(sample_rate * 3) else wav
                tmp_anc = tempfile.NamedTemporaryFile(suffix="_anchor.wav", delete=False)
                sf.write(tmp_anc.name, anchor_wav, sample_rate)
                voice_anchor_path = tmp_anc.name
            except Exception:
                voice_anchor_path = None

        faded_wav = apply_fade(wav, fade_samples=int(sample_rate * 0.01))
        audio_segments.append(faded_wav)
        dur_c = len(wav) / sample_rate
        chunk_durations.append(dur_c)

        # Jeda nafas manusiawi
        pause_sec = get_adaptive_pause_sec(clean_c)
        if i < num_chunks - 1:
            silence = np.zeros(int(sample_rate * pause_sec), dtype=np.float32)
            audio_segments.append(silence)

        prog_ratio = (i + 1) / num_chunks
        progress(prog_ratio, desc=f"Bagian [{i+1}/{num_chunks}] selesai ({dur_c:.1f}s)")

    progress(0.98, desc="Menggabungkan audio & membuat subtitle...")
    full_audio = np.concatenate(audio_segments) if len(audio_segments) > 1 else audio_segments[0]
    final_duration = len(full_audio) / sample_rate

    # Subtitle SRT & ASS Generation
    srt_lines = []
    curr_time = 0.0
    for idx, (c_text, dur) in enumerate(zip(chunks, chunk_durations)):
        s_start = format_timestamp_srt(curr_time)
        s_end = format_timestamp_srt(curr_time + dur)
        srt_lines.append(f"{idx + 1}\n{s_start} --> {s_end}\n{c_text}\n")
        p_sec = get_adaptive_pause_sec(c_text) if idx < len(chunks) - 1 else 0.0
        curr_time += dur + p_sec
    srt_content = "\n".join(srt_lines)

    ass_header = (
        "[Script Info]\nTitle: intiVoice Studio Subtitle\nScriptType: v4.00+\nWrapStyle: 0\nScaledBorderAndShadow: yes\nYCbCr Matrix: None\nPlayResX: 1080\nPlayResY: 1920\n\n"
        "[V4+ Styles]\nFormat: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding\n"
        "Style: Default,Arial,68,&H00FFFFFF,&H000000FF,&H00000000,&H80000000,-1,0,0,0,100,100,0,0,1,3,0,2,30,30,280,1\n\n"
        "[Events]\nFormat: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text\n"
    )
    ass_dialogues = []
    curr_time = 0.0
    for idx, (c_text, dur) in enumerate(zip(chunks, chunk_durations)):
        a_start = format_timestamp_ass(curr_time)
        a_end = format_timestamp_ass(curr_time + dur)
        ass_dialogues.append(f"Dialogue: 0,{a_start},{a_end},Default,,0,0,0,,{c_text}")
        p_sec = get_adaptive_pause_sec(c_text) if idx < len(chunks) - 1 else 0.0
        curr_time += dur + p_sec
    ass_content = ass_header + "\n".join(ass_dialogues)

    # Simpan Berkas Hasil
    base_name = f"intivoice_{int(time.time())}"
    wav_path = os.path.join(OUTPUTS_DIR, f"{base_name}.wav")
    srt_path = os.path.join(OUTPUTS_DIR, f"{base_name}.srt")
    ass_path = os.path.join(OUTPUTS_DIR, f"{base_name}.ass")

    sf.write(wav_path, full_audio, sample_rate, format="WAV")
    with open(srt_path, "w", encoding="utf-8") as fs:
        fs.write(srt_content)
    with open(ass_path, "w", encoding="utf-8") as fa:
        fa.write(ass_content)

    elapsed = time.time() - start_time
    rtf = elapsed / final_duration if final_duration > 0 else 0.0
    mode_tag = "Ultimate Clone 1:1" if is_ultimate_cloning else ("Timbre Clone" if ref_16k_path else "Voice Design")
    status_msg = f"🎉 Sintesis Sukses [{mode_tag}] | Render: {elapsed:.2f}s | Durasi Audio: {final_duration:.2f}s (48kHz) | RTF: {rtf:.2f}"
    progress(1.0, desc="Selesai 100%!")
    return wav_path, srt_path, ass_path, status_msg

# ==============================================================================
# CALLBACK TAB 2: PERCAKAPAN DUA SUARA (NATIVE IN-MEMORY)
# ==============================================================================
def synthesize_dialogue_gradio(script_text, spk1_name, spk1_preset, spk1_audio, spk2_name, spk2_preset, spk2_audio, default_pause, cfg_val, steps_val, progress=gr.Progress()):
    if not script_text or not script_text.strip():
        return None, None, None, "⚠️ Silakan masukkan naskah dialog percakapan terlebih dahulu!"

    name1 = (spk1_name.strip() or "Pembicara 1").lower()
    name2 = (spk2_name.strip() or "Pembicara 2").lower()
    prompt1 = PRESETS.get(spk1_preset, "")
    prompt2 = PRESETS.get(spk2_preset, "")
    ref1_path = prepare_reference_wav(spk1_audio) if spk1_audio else None
    ref2_path = prepare_reference_wav(spk2_audio) if spk2_audio else None

    # Parse script naskah
    raw_lines = [l.strip() for l in script_text.splitlines() if l.strip()]
    turns = []
    for line in raw_lines:
        spk, em, spoken = parse_dialogue_chunk(line)
        if not spk:
            spk = name1
            spoken = line
        turns.append({"speaker": spk, "emotion": em, "text": spoken})

    if not turns:
        return None, None, None, "⚠️ Format naskah dialog tidak terbaca! Gunakan format: Nama: Ucapan"

    start_time = time.time()
    progress(0.05, desc=f"Merender {len(turns)} giliran bicara di GPU...")

    turns_audio = []
    turns_metadata = []
    curr_offset = 0.0
    pause_between = float(default_pause or 0.40)

    for i, t in enumerate(turns):
        spk_raw = t["speaker"].lower()
        is_spk2 = (name2 in spk_raw) or (spk_raw.startswith(name2[:4]))
        active_prompt = prompt2 if is_spk2 else prompt1
        active_ref = ref2_path if is_spk2 else ref1_path

        em = t["emotion"].lower() if t["emotion"] else ""
        em_hint = EMOTION_THESAURUS.get(em, "")
        controls = []
        if em_hint:
            controls.append(em_hint)
        elif active_prompt:
            controls.append(active_prompt)
        prefix = f"({', '.join(controls)})" if controls else ""
        prompt_chunk = f"{prefix}{t['text']}"

        gen_k = {
            "text": prompt_chunk,
            "cfg_value": float(cfg_val or 2.0),
            "inference_timesteps": int(steps_val or 10),
            "retry_badcase": True,
        }
        if active_ref:
            gen_k["reference_wav_path"] = active_ref

        with torch.inference_mode():
            wav = model.generate(**gen_k)

        faded = apply_fade(wav, fade_samples=int(sample_rate * 0.01))
        turns_audio.append(faded)
        dur_t = len(wav) / sample_rate

        turns_metadata.append({
            "speaker": t["speaker"],
            "text": t["text"],
            "start": curr_offset,
            "end": curr_offset + dur_t,
        })
        curr_offset += dur_t + pause_between
        progress((i + 1) / len(turns), desc=f"Giliran [{i+1}/{len(turns)}] selesai ({t['speaker']})")

    # Assemble Dialogue Audio
    assembled = []
    for i, a in enumerate(turns_audio):
        assembled.append(a)
        if i < len(turns_audio) - 1:
            silence = np.zeros(int(sample_rate * pause_between), dtype=np.float32)
            assembled.append(silence)
    full_dlg = np.concatenate(assembled) if assembled else np.array([], dtype=np.float32)
    total_dur = len(full_dlg) / sample_rate

    # Subtitle Dual-Speaker
    srt_lines = []
    for idx, tm in enumerate(turns_metadata):
        s_st = format_timestamp_srt(tm["start"])
        s_en = format_timestamp_srt(tm["end"])
        srt_lines.append(f"{idx + 1}\n{s_st} --> {s_en}\n{tm['speaker']}: {tm['text']}\n")
    srt_dlg = "\n".join(srt_lines)

    ass_header = (
        "[Script Info]\nTitle: intiVoice Dialogue Subtitle\nScriptType: v4.00+\nWrapStyle: 0\nScaledBorderAndShadow: yes\nYCbCr Matrix: None\nPlayResX: 1080\nPlayResY: 1920\n\n"
        "[V4+ Styles]\nFormat: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding\n"
        "Style: Default,Arial,68,&H00FFFFFF,&H000000FF,&H00000000,&H80000000,-1,0,0,0,100,100,0,0,1,3,0,2,30,30,280,1\n\n"
        "[Events]\nFormat: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text\n"
    )
    ass_lines = []
    for tm in turns_metadata:
        a_st = format_timestamp_ass(tm["start"])
        a_en = format_timestamp_ass(tm["end"])
        ass_lines.append(f"Dialogue: 0,{a_st},{a_en},Default,,0,0,0,,{tm['speaker']}: {tm['text']}")
    ass_dlg = ass_header + "\n".join(ass_lines)

    dlg_base = f"dialogue_{int(time.time())}"
    wav_dlg_path = os.path.join(OUTPUTS_DIR, f"{dlg_base}.wav")
    srt_dlg_path = os.path.join(OUTPUTS_DIR, f"{dlg_base}.srt")
    ass_dlg_path = os.path.join(OUTPUTS_DIR, f"{dlg_base}.ass")

    sf.write(wav_dlg_path, full_dlg, sample_rate, format="WAV")
    with open(srt_dlg_path, "w", encoding="utf-8") as fs:
        fs.write(srt_dlg)
    with open(ass_dlg_path, "w", encoding="utf-8") as fa:
        fa.write(ass_dlg)

    elapsed = time.time() - start_time
    status_msg = f"🎭 Dialog Selesai! ({len(turns)} giliran) | Render: {elapsed:.2f}s | Durasi: {total_dur:.2f}s (48kHz)"
    progress(1.0, desc="Dialog Selesai!")
    return wav_dlg_path, srt_dlg_path, ass_dlg_path, status_msg

# ==============================================================================
# 5. TAMPILAN ANTARMUKA VISUAL GRADIO BLOCKS MODERN & PREMIUM (INTISARIAPPS)
# ==============================================================================
print("\n[3/3] 🌐 Meluncurkan Web Studio Publik (Premium IntisariApps UI)...")

custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap');
* { font-family: 'Inter', sans-serif !important; }
.gradio-container { max-width: 1240px !important; margin: 0 auto !important; }
.header-banner {
    background: linear-gradient(135deg, #071952 0%, #0D2F6D 50%, #1A73E8 100%);
    border: 1px solid rgba(255, 255, 255, 0.15);
    border-radius: 20px;
    padding: 24px 28px;
    margin-bottom: 20px;
    box-shadow: 0 12px 36px rgba(0, 0, 0, 0.35);
}
.badge-item {
    display: inline-flex;
    align-items: center;
    gap: 6px;
    padding: 5px 12px;
    border-radius: 9999px;
    font-size: 11px;
    font-weight: 600;
    margin-right: 8px;
    margin-top: 8px;
    background: rgba(255, 255, 255, 0.12);
    border: 1px solid rgba(255, 255, 255, 0.2);
    color: #FFFFFF;
}
.action-btn {
    background: linear-gradient(90deg, #1A73E8 0%, #00BFA5 100%) !important;
    color: #FFFFFF !important;
    border: none !important;
    border-radius: 12px !important;
    font-weight: 700 !important;
    font-size: 15px !important;
    box-shadow: 0 6px 20px rgba(26, 115, 232, 0.4) !important;
    transition: all 0.2s ease !important;
}
.action-btn:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 8px 25px rgba(0, 191, 165, 0.5) !important;
}
.panel-box {
    border: 1px solid rgba(255, 255, 255, 0.1);
    border-radius: 16px;
    padding: 16px;
}
"""

custom_theme = gr.themes.Soft(
    primary_hue="blue",
    secondary_hue="cyan",
    neutral_hue="slate",
)

with gr.Blocks(theme=custom_theme, css=custom_css, title="intiVoice AI Studio") as demo:
    gr.HTML(
        '''
        <div class="header-banner">
            <div style="display: flex; align-items: center; gap: 12px; margin-bottom: 8px;">
                <span style="font-size: 32px;">🎙️</span>
                <div>
                    <h1 style="margin: 0; font-size: 24px; font-weight: 800; color: #FFFFFF; letter-spacing: -0.5px;">
                        intiVoice AI <span style="color: #00D2B4; font-weight: 700;">Studio Suara Pintar</span>
                    </h1>
                    <p style="margin: 3px 0 0 0; font-size: 13px; color: #E2E8F0; opacity: 0.9;">
                        Sintesis Suara AI Natural, Voice Design & Ultimate Voice Cloning 1:1 (48kHz Studio Audio)
                    </p>
                </div>
            </div>
            <div style="margin-top: 14px; display: flex; flex-wrap: wrap;">
                <span class="badge-item"><span style="color: #00FF88;">●</span> GPU: Tesla T4 (Online)</span>
                <span class="badge-item">⚡ 48.000 Hz WAV</span>
                <span class="badge-item">🔒 Voice Anchor DNA</span>
                <span class="badge-item">🗣️ Human-Cadence (240c)</span>
                <span class="badge-item">🎭 13 Emosi Vokal</span>
                <span class="badge-item">🎬 CapCut .SRT & .ASS</span>
                <span class="badge-item" style="background: rgba(0,210,180,0.2); border-color: #00D2B4;">🚀 In-Memory Native</span>
            </div>
        </div>
        '''
    )

    with gr.Tabs():
        # ==========================================================
        # TAB 1: STUDIO VOKAL TUNGGAL
        # ==========================================================
        with gr.TabItem("🎙️ Studio Vokal Tunggal (TTS & Cloning)"):
            with gr.Row():
                with gr.Column(scale=5):
                    text_input = gr.Textbox(
                        label="📝 Naskah Teks (Dukungan Naskah Panjang Smart Chunking)",
                        placeholder="Ketik naskah cerita, artikel berita, atau naskah promosi di sini...",
                        lines=6,
                        value="Halo semuanya! Selamat datang di era baru kecerdasan buatan bersama intiVoice. Setiap kalimat dipotong alami dengan ritme nafas manusiawi dan kualitas audio studio 48 kilohertz."
                    )
                    with gr.Row():
                        preset_dropdown = gr.Dropdown(
                            choices=list(PRESETS.keys()),
                            value=list(PRESETS.keys())[0],
                            label="🎨 Karakter Suara Resmi"
                        )
                        emotion_dropdown = gr.Dropdown(
                            choices=EMOTIONS,
                            value=EMOTIONS[0],
                            label="🎭 Kamus 13 Emosi Vokal"
                        )

                    custom_prompt_input = gr.Textbox(
                        label="✏️ Kustom Voice Design Prompt",
                        placeholder="Contoh: A cheerful young Indonesian mother, soft and warm tone...",
                        lines=2,
                        visible=False
                    )

                    with gr.Accordion("🎙️ Ultimate Voice Cloning (Kloning Suara 1:1)", open=False):
                        gr.HTML(
                            '''
                            <div style="background: rgba(26, 115, 232, 0.08); border-left: 4px solid #1A73E8; padding: 10px 14px; border-radius: 8px; margin-bottom: 12px; font-size: 12px; color: #CBD5E1;">
                                <b>💡 Naskah Rekaman Ideal (Baca saat merekam 6-8 detik):</b><br/>
                                <i>"Halo semuanya! Selamat datang di era baru kecerdasan buatan, di mana setiap cerita dan suara memiliki kehangatan yang nyata."</i>
                            </div>
                            '''
                        )
                        ref_audio_input = gr.Audio(label="Audio Referensi Kloning (WAV/MP3 - Rekaman Bersih 6-8s)", type="filepath")
                        ref_transcript_input = gr.Textbox(
                            label="Transkrip Kata Persis dari Audio Acuan (Wajib untuk Kloning 1:1)",
                            placeholder="Ketik kata-kata persis yang diucapkan pada rekaman audio di atas...",
                            lines=2
                        )

                    with gr.Accordion("⚙️ Parameter Vokal & Performa (Tuning)", open=False):
                        with gr.Row():
                            speed_slider = gr.Slider(minimum=0.75, maximum=1.35, value=1.0, step=0.05, label="Kecepatan Bicara (Speed)")
                            cfg_slider = gr.Slider(minimum=1.0, maximum=4.0, value=2.0, step=0.1, label="CFG Scale (Kepatuhan Teks)")
                        with gr.Row():
                            steps_slider = gr.Slider(minimum=5, maximum=25, value=10, step=1, label="Inference Timesteps")
                            seed_input = gr.Number(value=428190, label="Fixed Seed (DNA Karakter Suara)")

                    btn_submit_single = gr.Button("🚀 Mulai Sintesis Suara (48kHz WAV)", elem_classes=["action-btn"], size="lg")

                with gr.Column(scale=4):
                    audio_output = gr.Audio(label="🔊 Pemutar Audio Hasil (48kHz Studio Quality)", type="filepath", interactive=False)
                    with gr.Row():
                        srt_output = gr.File(label="🎬 Unduh Subtitle CapCut (.SRT)", interactive=False)
                        ass_output = gr.File(label="✨ Unduh Subtitle Karaoke (.ASS)", interactive=False)
                    status_output = gr.Textbox(label="📊 Status Sintesis Real-Time", interactive=False)

            def on_preset_change(c):
                return gr.update(visible=(c == "⚡ Kustom (Ketik Prompt Karakter Sendiri)"))
            preset_dropdown.change(fn=on_preset_change, inputs=[preset_dropdown], outputs=[custom_prompt_input])

            btn_submit_single.click(
                fn=synthesize_single_gradio,
                inputs=[
                    text_input,
                    preset_dropdown,
                    custom_prompt_input,
                    emotion_dropdown,
                    ref_audio_input,
                    ref_transcript_input,
                    cfg_slider,
                    steps_slider,
                    seed_input,
                    speed_slider
                ],
                outputs=[audio_output, srt_output, ass_output, status_output]
            )

        # ==========================================================
        # TAB 2: PERCAKAPAN DUA SUARA (DUAL-SPEAKER DIALOGUE)
        # ==========================================================
        with gr.TabItem("👥 Percakapan 2 Suara (Dual-Speaker Dialogue)"):
            with gr.Row():
                with gr.Column(scale=5):
                    dialogue_script = gr.Textbox(
                        label="📜 Naskah Naskah Percakapan (Format: Nama: Ucapan)",
                        placeholder="Rian: Halo Sari, bagaimana kabar proyek AI suara kita hari ini?\nSari: (tersenyum) Luar biasa Rian! Kualitasnya sangat jernih dan natural.\nRian: (tertawa) Betul sekali, jeda nafasnya terdengar seperti manusia asli!",
                        lines=7,
                        value="Rian: Halo Sari, bagaimana kabar proyek AI suara kita hari ini?\nSari: (tersenyum) Luar biasa Rian! Kualitasnya sangat jernih dan natural.\nRian: (tertawa) Betul sekali, jeda nafasnya terdengar seperti manusia asli!"
                    )
                    with gr.Row():
                        with gr.Column():
                            gr.Markdown("### 👤 Pembicara 1")
                            spk1_name = gr.Textbox(label="Nama Karakter 1", value="Rian")
                            spk1_preset = gr.Dropdown(choices=list(PRESETS.keys()), value=list(PRESETS.keys())[1], label="Preset Karakter 1")
                            spk1_audio = gr.Audio(label="Audio Kloning 1 (Opsional)", type="filepath")
                        with gr.Column():
                            gr.Markdown("### 👤 Pembicara 2")
                            spk2_name = gr.Textbox(label="Nama Karakter 2", value="Sari")
                            spk2_preset = gr.Dropdown(choices=list(PRESETS.keys()), value=list(PRESETS.keys())[0], label="Preset Karakter 2")
                            spk2_audio = gr.Audio(label="Audio Kloning 2 (Opsional)", type="filepath")

                    with gr.Accordion("⚙️ Pengaturan Jeda & Parameter Dialog", open=False):
                        with gr.Row():
                            pause_slider = gr.Slider(minimum=0.20, maximum=1.00, value=0.40, step=0.05, label="Jeda Antar Pembicara (Detik)")
                            cfg_dlg_slider = gr.Slider(minimum=1.0, maximum=4.0, value=2.0, step=0.1, label="CFG Scale")
                            steps_dlg_slider = gr.Slider(minimum=5, maximum=25, value=10, step=1, label="Inference Steps")

                    btn_submit_dlg = gr.Button("🎭 Render Percakapan Dua Suara (48kHz)", elem_classes=["action-btn"], size="lg")

                with gr.Column(scale=4):
                    dlg_audio_out = gr.Audio(label="🔊 Audio Percakapan Utuh (48kHz WAV)", type="filepath", interactive=False)
                    with gr.Row():
                        dlg_srt_out = gr.File(label="🎬 Subtitle Dialog (.SRT)", interactive=False)
                        dlg_ass_out = gr.File(label="✨ Subtitle Dialog (.ASS)", interactive=False)
                    dlg_status_out = gr.Textbox(label="📊 Status Percakapan Real-Time", interactive=False)

            btn_submit_dlg.click(
                fn=synthesize_dialogue_gradio,
                inputs=[
                    dialogue_script,
                    spk1_name,
                    spk1_preset,
                    spk1_audio,
                    spk2_name,
                    spk2_preset,
                    spk2_audio,
                    pause_slider,
                    cfg_dlg_slider,
                    steps_dlg_slider
                ],
                outputs=[dlg_audio_out, dlg_srt_out, dlg_ass_out, dlg_status_out]
            )

        # ==========================================================
        # TAB 3: PANDUAN PENGGUNAAN & TIPS REKAMAN
        # ==========================================================
        with gr.TabItem("💡 Panduan & Tips Rekaman Ideal"):
            gr.Markdown(
                """
                ### 🎙️ Panduan Kloning Suara 1:1 Presisi Tinggi (Ultimate Cloning)
                1. **Durasi Rekaman:** Rekam suara Anda selama **6 hingga 8 detik**. Jangan terlalu pendek (< 3s) dan jangan terlalu panjang (> 12s).
                2. **Lingkungan Hening:** Pastikan tidak ada suara musik latar, dengung kipas angin, AC, atau gema ruangan (*room reverb*).
                3. **Naskah Standar Baku:** Baca naskah standar dengan santai dan artikulasi jelas:
                   > *"Halo semuanya! Selamat datang di era baru kecerdasan buatan, di mana setiap cerita dan suara memiliki kehangatan yang nyata."
                4. **Transkrip Acuan:** Pastikan kolom transkrip diisi persis sama dengan kata-kata yang Anda ucapkan.

                ---

                ### 🎭 Cara Menggunakan 13 Kamus Emosi Vokal
                Anda dapat memilih emosi dari dropdown atau mengetik tag emosi di dalam tanda kurung di awal kalimat:
                - `(berbisik)` : Nada suara pelan, intim, dan berbisik lembut.
                - `(tertawa)` : Suara ceria diiringi kekehan tawa santai.
                - `(senang)` / `(bahagia)` : Vokal hangat, riang, dan positif.
                - `(sedih)` : Intonasi pelan dan emosional.
                - `(marah)` : Artikulasi tegas, keras, dan bertenaga.

                ---

                ### 🎬 Memasukkan Subtitle ke CapCut & Premiere Pro
                - **File .SRT**: Tarik langsung berkas `.srt` ke timeline CapCut Desktop atau Premiere Pro. Teks otomatis sinkron dengan gelombang audio.
                - **File .ASS**: Berisi format karaoke dengan safe zone 9:16 untuk video pendek TikTok, YouTube Shorts, dan Reels.
                """
            )

print("\n================================================================================")
print("🚀 Antarmuka Web Studio Siap! Link Publik (gradio.live) akan muncul di bawah:")
print("================================================================================")
demo.queue().launch(share=True, inline=True)
